In [ ]:
import tkinter as tk
from tkinter import messagebox

import vault
from strength import check_strength


In [ ]:
def get_valid_password():
    while True:
        pw = input("Enter a password : ")
        score = check_strength(pw)
        print("Strength score =", score)
        if score >= 6:
            return pw
        print("Password too weak, try again")


def add_user():
    print("_____ ADDING NEW USER _____")
    uname = input("Choose a username: ")

    if vault.check_user(uname):
        print("User already exists")
        return

    print("Create a strong password")
    while True:
        pw = get_valid_password()
        pw2 = input("Confirm password: ")
        if pw == pw2:
            break
        print("Passwords do not match, try again")

    vault.register(uname, pw)
    print("User created")


def login_cli():
    uname = input("Username: ")
    pw = input("Password: ")
    ok, key = vault.login(uname, pw)
    if ok:
        return uname, key
    return None, None


def upload_file(username, key):
    print("_____ UPLOAD FILE _____")
    fname = input("Enter file name to store: ")

    if fname in vault.list_files(username):
        print("File already exists, choose another name")
        return

    text = input("Enter file contents: ")
    token = vault.encrypt(text.encode(), key)
    vault.save_encrypted_file(username, fname, token)
    print("File encrypted and saved")


def view_file(username, key):
    print("_____ VIEW FILE _____")
    fname = input("Enter file name: ")
    try:
        token = vault.load_encrypted_file(username, fname)
        data = vault.decrypt(token, key)
        print("Decrypted text:")
        print(data.decode())
    except Exception:
        print("Could not open or decrypt file")


def delete_user_file(username):
    print("_____ DELETE FILE _____")
    fname = input("Enter file name: ")
    if vault.delete_file(username, fname):
        print("File deleted")
    else:
        print("File not found")


def show_files(username):
    print("_____ USER FILES _____")
    for fname in vault.list_files(username):
        print(fname)


def user_menu(username, key):
    print("Welcome", username)
    while True:
        print("")
        print("Choose an option:")
        print("1. Upload file")
        print("2. View file")
        print("3. Delete file")
        print("4. List files")
        print("5. Logout")

        choice = input("Enter choice: ")

        if choice == "1":
            upload_file(username, key)
        elif choice == "2":
            view_file(username, key)
        elif choice == "3":
            delete_user_file(username)
        elif choice == "4":
            show_files(username)
        elif choice == "5":
            break
        else:
            print("Invalid choice")


def main_menu():
    while True:
        print("\n_____ MAIN MENU _____")
        print("1. Register")
        print("2. Login")
        print("3. Exit")

        choice = input("Enter choice: ")

        if choice == "1":
            add_user()
        elif choice == "2":
            print("_____ LOGIN _____")
            username, key = login_cli()
            if username:
                user_menu(username, key)
            else:
                print("Login failed")
        elif choice == "3":
            break
        else:
            print("Invalid choice")


In [ ]:
def gui_list_files(username):
    files = vault.list_files(username)

    win = tk.Toplevel()
    win.title("Your Files")
    win.geometry("300x250")

    tk.Label(win, text="Files for " + username, font=("Arial", 12)).pack(pady=5)

    if not files:
        tk.Label(win, text="No files found").pack()
        return

    listbox = tk.Listbox(win, width=40)
    listbox.pack(pady=10)

    for fname in files:
        listbox.insert(tk.END, fname)


def gui_view_file(username, key):
    popup = tk.Toplevel()
    popup.title("View File")
    popup.geometry("260x150")

    tk.Label(popup, text="Enter file name:").pack(pady=5)
    entry = tk.Entry(popup)
    entry.pack(pady=5)

    def load():
        fname = entry.get()
        try:
            token = vault.load_encrypted_file(username, fname)
            data = vault.decrypt(token, key)
            text = data.decode()

            viewer = tk.Toplevel()
            viewer.title(fname)
            viewer.geometry("360x300")

            tk.Label(viewer, text=f"Contents of {fname}", font=("Arial", 12)).pack(pady=5)

            txt = tk.Text(viewer, wrap="word")
            txt.pack(expand=True, fill="both")
            txt.insert("1.0", text)

        except Exception:
            messagebox.showerror("Error", "Could not open or decrypt file")

        popup.destroy()

    tk.Button(popup, text="Open", width=15, command=load).pack(pady=10)


def gui_register():
    win = tk.Toplevel()
    win.title("Register New User")
    win.geometry("320x280")

    tk.Label(win, text="Choose a username:").pack(pady=5)
    entry_user = tk.Entry(win)
    entry_user.pack(pady=5)

    tk.Label(win, text="Choose a strong password:").pack(pady=5)
    entry_pass = tk.Entry(win, show="*")
    entry_pass.pack(pady=5)

    tk.Label(win, text="Confirm password:").pack(pady=5)
    entry_pass2 = tk.Entry(win, show="*")
    entry_pass2.pack(pady=5)

    def submit_register():
        username = entry_user.get().strip()
        pw1 = entry_pass.get()
        pw2 = entry_pass2.get()

        if username == "" or pw1 == "" or pw2 == "":
            messagebox.showerror("Error", "All fields are required")
            return

        if pw1 != pw2:
            messagebox.showerror("Error", "Passwords do not match")
            return

        if vault.check_user(username):
            messagebox.showerror("Error", "User already exists")
            return

        score = check_strength(pw1)
        if score < 6:
            messagebox.showerror("Error", f"Password too weak! Strength score: {score}")
            return

        vault.register(username, pw1)

        messagebox.showinfo("Success", f"User '{username}' created successfully!")
        win.destroy()

    tk.Button(win, text="Register", width=20, command=submit_register).pack(pady=15)
    tk.Button(win, text="Cancel", width=20, command=win.destroy).pack()


def gui_upload_file(username, key):
    win = tk.Toplevel()
    win.title("Upload File")
    win.geometry("350x300")

    tk.Label(win, text="File Name:").pack(pady=4)
    entry_name = tk.Entry(win)
    entry_name.pack(pady=4)

    tk.Label(win, text="File Contents:").pack(pady=4)
    text_box = tk.Text(win, height=8, width=30)
    text_box.pack(pady=4)

    def save():
        fname = entry_name.get()
        contents = text_box.get("1.0", tk.END).strip()

        if fname.strip() == "":
            messagebox.showerror("Error", "File name cannot be empty")
            return

        token = vault.encrypt(contents.encode(), key)
        vault.save_encrypted_file(username, fname, token)

        messagebox.showinfo("Done", "File encrypted and saved.")
        win.destroy()

    tk.Button(win, text="Upload", width=15, command=save).pack(pady=10)


def gui_delete_file(username):
    popup = tk.Toplevel()
    popup.title("Delete File")
    popup.geometry("260x150")

    tk.Label(popup, text="Enter file name:").pack(pady=5)
    entry = tk.Entry(popup)
    entry.pack(pady=5)

    def delete_now():
        fname = entry.get()
        if vault.delete_file(username, fname):
            messagebox.showinfo("Done", "File deleted.")
        else:
            messagebox.showerror("Error", "File not found.")
        popup.destroy()

    tk.Button(popup, text="Delete", width=15, command=delete_now).pack(pady=10)


In [ ]:
def gui_user_menu(username, key):
    BG_COLOR = "#2C3E50"
    FG_COLOR = "#ECF0F1"
    BTN_COLOR = "#2980B9"
    BTN_HOVER = "#3498DB"
    FONT_TITLE = ("Helvetica", 16, "bold")
    FONT_BTN = ("Helvetica", 11, "bold")
    PAD_Y = 8

    def style_button(btn):
        btn.config(bg=BTN_COLOR, fg=FG_COLOR, font=FONT_BTN, activebackground=BTN_HOVER, relief="raised")
        btn.bind("<Enter>", lambda e: btn.config(bg=BTN_HOVER))
        btn.bind("<Leave>", lambda e: btn.config(bg=BTN_COLOR))

    win = tk.Toplevel()
    win.title(f"User Menu - {username}")
    win.configure(bg=BG_COLOR)
    win.geometry("350x360")

    tk.Label(win, text=f"Welcome {username}", font=FONT_TITLE, fg=FG_COLOR, bg=BG_COLOR).pack(pady=15)
    tk.Label(win, text="Choose an option:", fg=FG_COLOR, bg=BG_COLOR).pack(pady=PAD_Y)

    buttons = [
        ("1. Upload File", lambda: gui_upload_file(username, key)),
        ("2. View File", lambda: gui_view_file(username, key)),
        ("3. Delete File", lambda: gui_delete_file(username)),
        ("4. List Files", lambda: gui_list_files(username)),
        ("5. Logout", win.destroy),
    ]

    for text, cmd in buttons:
        b = tk.Button(win, text=text, width=25, command=cmd)
        style_button(b)
        b.pack(pady=PAD_Y)


def gui_main_menu():
    BG_COLOR = "#2C3E50"
    FG_COLOR = "#ECF0F1"
    BTN_COLOR = "#2980B9"
    BTN_HOVER = "#3498DB"
    FONT_TITLE = ("Helvetica", 16, "bold")
    FONT_BTN = ("Helvetica", 11, "bold")
    PAD_Y = 8

    def style_button(btn):
        btn.config(bg=BTN_COLOR, fg=FG_COLOR, font=FONT_BTN, activebackground=BTN_HOVER, relief="raised")
        btn.bind("<Enter>", lambda e: btn.config(bg=BTN_HOVER))
        btn.bind("<Leave>", lambda e: btn.config(bg=BTN_COLOR))

    root = tk.Tk()
    root.title("MAIN MENU")
    root.configure(bg=BG_COLOR)
    root.geometry("350x300")

    tk.Label(root, text="_____ MAIN MENU _____", font=FONT_TITLE, fg=FG_COLOR, bg=BG_COLOR).pack(pady=15)

    btn_reg = tk.Button(root, text="1. Register", width=25, command=gui_register)
    style_button(btn_reg)
    btn_reg.pack(pady=PAD_Y)

    def on_login():
        win = tk.Toplevel(root)
        win.title("LOGIN")
        win.configure(bg=BG_COLOR)
        win.geometry("350x220")

        tk.Label(win, text="Login", font=FONT_TITLE, fg=FG_COLOR, bg=BG_COLOR).pack(pady=10)
        tk.Label(win, text="Username:", fg=FG_COLOR, bg=BG_COLOR).pack(pady=PAD_Y)
        entry_user = tk.Entry(win)
        entry_user.pack()

        tk.Label(win, text="Password:", fg=FG_COLOR, bg=BG_COLOR).pack(pady=PAD_Y)
        entry_pass = tk.Entry(win, show="*")
        entry_pass.pack()

        def submit_login():
            username = entry_user.get()
            ok, key = vault.login(username, entry_pass.get())
            if ok:
                messagebox.showinfo("Success", f"Login successful. Welcome, {username}!")
                win.destroy()
                gui_user_menu(username, key)
            else:
                messagebox.showerror("Error", "Login failed")

        btn_login = tk.Button(win, text="Login", width=20, command=submit_login)
        style_button(btn_login)
        btn_login.pack(pady=PAD_Y)

        btn_cancel = tk.Button(win, text="Cancel", width=20, command=win.destroy)
        style_button(btn_cancel)
        btn_cancel.pack(pady=PAD_Y)

    btn_login_main = tk.Button(root, text="2. Login", width=25, command=on_login)
    style_button(btn_login_main)
    btn_login_main.pack(pady=PAD_Y)

    def on_exit():
        root.destroy()

    btn_exit = tk.Button(root, text="3. Exit", width=25, command=on_exit)
    style_button(btn_exit)
    btn_exit.pack(pady=PAD_Y + 5)

    root.mainloop()


In [ ]:
gui_main_menu()
